In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [ ]:
spark = SparkSession.builder.getOrCreate()
spark.stop()

spark = SparkSession.builder \
    .appName("GFN-EDA") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.legacy.parquetNanosAsLong", "true") \
    .getOrCreate()

print(f"Master: {spark.sparkContext.master}")
print(f"Spark UI URL: {spark.sparkContext.uiWebUrl}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 02:57:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Master: local[*]
Spark UI URL: http://9898a2975188:4040


In [4]:
users = spark.read.parquet("/home/spark/work/data/raw/users.parquet")
sessions = spark.read.parquet("/home/spark/work/data/raw/session_logs.parquet")
games = spark.read.parquet("/home/spark/work/data/raw/game_catalog.parquet")
sub_events = spark.read.parquet("/home/spark/work/data/raw/subscription_events.parquet")
payments = spark.read.parquet("/home/spark/work/data/raw/payments.parquet")

User's basic EDA

In [5]:
for name, df in [("users", users), ("sessions", sessions), ("games", games),
                  ("sub_events", sub_events), ("payments", payments)]:
    print(f"=== {name}: {df.count():,} rows, {len(df.columns)} cols ===")
    df.printSchema()

=== users: 50,000 rows, 9 cols ===
root
 |-- user_id: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- subscription_tier: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- region: string (nullable = true)
 |-- referral_source: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- persona: string (nullable = true)

=== sessions: 3,553,650 rows, 15 cols ===
root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- game_id: string (nullable = true)
 |-- start_time: timestamp_ntz (nullable = true)
 |-- end_time: timestamp_ntz (nullable = true)
 |-- stream_resolution: string (nullable = true)
 |-- avg_latency_ms: double (nullable = true)
 |-- avg_fps: double (nullable = true)
 |-- total_frame_drops: long (nullable = true)
 |-- disconnect_count: long (nullable = true)
 |-- input_lag_ms: double (nullable = true)
 |-- avg_bitrate_mbps: double (nullable = true)
 |

In [5]:
users.groupBy("persona").count().orderBy("count", ascending=False).show()

+--------------+-----+
|       persona|count|
+--------------+-----+
|       regular|17409|
|        casual|15181|
|      hardcore| 9981|
|about_to_churn| 7429|
+--------------+-----+



In [6]:
users.groupBy("subscription_tier").count().orderBy("count", ascending=False).show()

+-----------------+-----+
|subscription_tier|count|
+-----------------+-----+
|         priority|19549|
|             free|18321|
|         ultimate|12130|
+-----------------+-----+



In [7]:
users.groupBy("persona") \
    .pivot("subscription_tier") \
    .count() \
    .show()

+--------------+-----+--------+--------+
|       persona| free|priority|ultimate|
+--------------+-----+--------+--------+
|        casual|10589|    3876|     716|
|      hardcore|  502|    3563|    5916|
|       regular| 3512|    9526|    4371|
|about_to_churn| 3718|    2584|    1127|
+--------------+-----+--------+--------+



In [8]:
for col in ["device_type", "region", "referral_source", "age_group", "gender"]:
    print(f"\n--- {col} ---")
    users.groupBy(col).count().orderBy("count", ascending=False).show(truncate=False)


--- device_type ---
+-----------+-----+
|device_type|count|
+-----------+-----+
|PC         |22590|
|Mobile     |9983 |
|Mac        |7516 |
|Chromebook |4971 |
|SHIELD     |4940 |
+-----------+-----+


--- region ---
+--------------+-----+
|region        |count|
+--------------+-----+
|New_Taipei    |8510 |
|Kaohsiung     |5979 |
|Taichung      |5953 |
|Taipei        |5444 |
|Taoyuan       |4967 |
|Tainan        |3898 |
|Changhua      |2540 |
|Pingtung      |1623 |
|Hsinchu_City  |1561 |
|Yunlin        |1521 |
|Hsinchu_County|1050 |
|Miaoli        |1035 |
|Nantou        |1008 |
|Yilan         |996  |
|Keelung       |982  |
|Chiayi_County |955  |
|Hualien       |522  |
|Chiayi_City   |488  |
|Taitung       |471  |
|Penghu        |252  |
+--------------+-----+
only showing top 20 rows


--- referral_source ---
+---------------+-----+
|referral_source|count|
+---------------+-----+
|organic        |19991|
|ad             |12594|
|friend_referral|10038|
|bundled        |7377 |
+----------

In [9]:
users.withColumn("signup_month", F.month(F.col("signup_date"))) \
    .groupBy("signup_month").count().orderBy("signup_month").show()

+------------+-----+
|signup_month|count|
+------------+-----+
|           1| 5202|
|           2| 4297|
|           3| 3201|
|           4| 3124|
|           5| 3592|
|           6| 3902|
|           7| 5675|
|           8| 5697|
|           9| 3860|
|          10| 3127|
|          11| 3468|
|          12| 4855|
+------------+-----+



session basic EDA

In [7]:
sessions.select(
    F.min("start_time").alias("earliest"),
    F.max("start_time").alias("latest"),
    F.count("session_id").alias("total_sessions"),
    F.countDistinct("user_id").alias("unique_users"),
    F.countDistinct("game_id").alias("unique_games"),
).show(truncate=False)

+-------------------+-------------------+--------------+------------+------------+
|earliest           |latest             |total_sessions|unique_users|unique_games|
+-------------------+-------------------+--------------+------------+------------+
|2024-01-01 00:00:00|2024-03-24 23:59:00|3553650       |50000       |200         |
+-------------------+-------------------+--------------+------------+------------+



In [8]:
sess_with_dur = sessions.withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)

sess_with_dur.select(
    F.mean("duration_min").alias("mean"),
    F.stddev("duration_min").alias("std"),
    F.expr("percentile_approx(duration_min, 0.5)").alias("p50"),
    F.expr("percentile_approx(duration_min, 0.95)").alias("p95"),
    F.expr("percentile_approx(duration_min, 0.99)").alias("p99"),
    F.max("duration_min").alias("max"),
).show()

+------------------+-----------------+-----------------+------------------+------------------+-----------------+
|              mean|              std|              p50|               p95|               p99|              max|
+------------------+-----------------+-----------------+------------------+------------------+-----------------+
|110.07204852756855|74.84534356465736|98.33333333333333|207.16666666666666|451.96666666666664|719.9833333333333|
+------------------+-----------------+-----------------+------------------+------------------+-----------------+



In [9]:
# Join persona info
sess_persona = sess_with_dur.join(
    users.select("user_id", "persona"), on="user_id"
)

sess_persona.groupBy("persona").agg(
    F.count("session_id").alias("total_sessions"),
    F.mean("duration_min").alias("avg_duration"),
    F.mean("avg_latency_ms").alias("avg_latency"),
    F.mean("avg_fps").alias("avg_fps"),
    F.mean("disconnect_count").alias("avg_disconnects"),
).show()

+--------------+--------------+------------------+------------------+------------------+-------------------+
|       persona|total_sessions|      avg_duration|       avg_latency|           avg_fps|    avg_disconnects|
+--------------+--------------+------------------+------------------+------------------+-------------------+
|        casual|        320095| 46.62416636727642|  50.3866145987913|47.338425467439386| 0.3008825504928224|
|      hardcore|       1695907|145.91826705905996|31.509288893789183| 55.25904799024881| 0.2998366066063764|
|       regular|       1207923| 87.68456164010475|38.386077754955565|  52.7380597107602|0.30026334460060783|
|about_to_churn|        329725| 69.31037511057204|45.497626810220765| 49.55514474183063|0.49813632572598376|
+--------------+--------------+------------------+------------------+------------------+-------------------+



In [12]:
import datetime

OBS_START = datetime.date(2024, 1, 1)

sess_weekly = sess_persona.withColumn(
    "week_num",
    F.floor(F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7) + 1
)

# Average sessions per user per week, grouped by persona
decay_check = sess_weekly.groupBy("persona", "week_num") \
    .agg(F.countDistinct("user_id").alias("active_users"),
         F.count("session_id").alias("sessions")) \
    .withColumn("sessions_per_user", F.col("sessions") / F.col("active_users")) \
    .orderBy("persona", "week_num")

decay_check.filter(F.col("persona") == "about_to_churn").show(12)
decay_check.filter(F.col("persona") == "hardcore").show(12)

+--------------+--------+------------+--------+------------------+
|       persona|week_num|active_users|sessions| sessions_per_user|
+--------------+--------+------------+--------+------------------+
|about_to_churn|       1|        7162|   31701| 4.426277576096062|
|about_to_churn|       2|        7148|   31132| 4.355344152210408|
|about_to_churn|       3|        7156|   31232|4.3644494130799325|
|about_to_churn|       4|        7168|   31145| 4.345005580357143|
|about_to_churn|       5|        7137|   31292| 4.384475269721172|
|about_to_churn|       6|        7136|   31149| 4.365050448430493|
|about_to_churn|       7|        7145|   31107|  4.35367389783065|
|about_to_churn|       8|        7358|   56138|  7.62951889100299|
|about_to_churn|       9|        7299|   35754| 4.898479243732018|
|about_to_churn|      10|        6034|   12430| 2.059993370898243|
|about_to_churn|      11|        3865|    5563| 1.439327296248383|
|about_to_churn|      12|         968|    1082|1.1177685950413

+--------+--------+------------+--------+------------------+
| persona|week_num|active_users|sessions| sessions_per_user|
+--------+--------+------------+--------+------------------+
|hardcore|       1|        9981|  126958| 12.71996793908426|
|hardcore|       2|        9981|  126443| 12.66836990281535|
|hardcore|       3|        9980|  126368|12.662124248496994|
|hardcore|       4|        9981|  126371| 12.66115619677387|
|hardcore|       5|        9981|  126379|12.661957719667368|
|hardcore|       6|        9981|  126283|12.652339444945396|
|hardcore|       7|        9981|  126133|12.637310890692316|
|hardcore|       8|        9981|  221528|22.194970443843303|
|hardcore|       9|        9981|  210602|21.100290552048893|
|hardcore|      10|        9981|  126340|12.658050295561567|
|hardcore|      11|        9980|  126275|12.652805611222444|
|hardcore|      12|        9981|  126227|12.646728784690913|
+--------+--------+------------+--------+------------------+



In [10]:
sessions.groupBy("exit_type").count().orderBy("count", ascending=False).show()

sess_persona.groupBy("persona").pivot("exit_type") \
    .agg(F.count("session_id")).show()

+----------+-------+
| exit_type|  count|
+----------+-------+
|    normal|2679751|
|disconnect| 542732|
|   timeout| 168178|
|     crash| 162989|
+----------+-------+



+--------------+-----+----------+-------+-------+
|       persona|crash|disconnect| normal|timeout|
+--------------+-----+----------+-------+-------+
|        casual|14287|     47143| 243495|  15170|
|      hardcore|76110|    250043|1290585|  79169|
|       regular|54098|    178108| 919295|  56422|
|about_to_churn|18494|     67438| 226376|  17417|
+--------------+-----+----------+-------+-------+



In [13]:
sess_weekly = sess_persona.withColumn(
    "week_num",
    F.floor(F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7) + 1
)

sess_weekly.filter(F.col("week_num") >= 8) \
    .groupBy("persona").pivot("exit_type").agg(F.count("session_id")).show()

+--------------+-----+----------+------+-------+
|       persona|crash|disconnect|normal|timeout|
+--------------+-----+----------+------+-------+
|        casual| 6835|     22895|117571|   7353|
|      hardcore|36336|    119744|617201|  37691|
|       regular|25986|     85682|442472|  27563|
|about_to_churn| 8685|     35326| 59840|   7116|
+--------------+-----+----------+------+-------+



In [14]:
sess_tier = sess_with_dur.join(users.select("user_id", "subscription_tier"), on="user_id")

sess_tier.filter(F.col("subscription_tier") == "free") \
    .select(F.max("duration_min").alias("max_free_dur")).show()

+-----------------+
|     max_free_dur|
+-----------------+
|719.9833333333333|
+-----------------+



Game Catalog EDA

In [15]:
games.groupBy("popularity_tier").count().orderBy("popularity_tier").show()
games.groupBy("genre").count().orderBy("count", ascending=False).show()

+---------------+-----+
|popularity_tier|count|
+---------------+-----+
|              A|   38|
|              B|   63|
|              C|   92|
|              S|    7|
+---------------+-----+

+-------------+-----+
|        genre|count|
+-------------+-----+
|   Simulation|   33|
|     Strategy|   31|
|       Casual|   30|
|          RPG|   24|
|       Racing|   22|
|          FPS|   22|
|       Sports|   20|
|Battle_Royale|   18|
+-------------+-----+



In [16]:
sessions.join(games.select("game_id", "popularity_tier"), on="game_id") \
    .groupBy("popularity_tier").count().orderBy("popularity_tier").show()

+---------------+-------+
|popularity_tier|  count|
+---------------+-------+
|              A|1412106|
|              B| 937153|
|              C| 683828|
|              S| 520563|
+---------------+-------+



Subscription Events EDA

In [17]:
sub_events.groupBy("event_type").count().orderBy("count", ascending=False).show()

# Per persona
sub_events.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").pivot("event_type").agg(F.count("event_id")).show()

+----------+-----+
|event_type|count|
+----------+-----+
|     renew|25630|
|    cancel| 5377|
|   upgrade| 3768|
| downgrade| 3763|
+----------+-----+

+--------------+------+---------+-----+-------+
|       persona|cancel|downgrade|renew|upgrade|
+--------------+------+---------+-----+-------+
|        casual|  1487|      815| 6129|   1416|
|      hardcore|    87|      175| 7928|    191|
|       regular|   878|     1475|10451|   1982|
|about_to_churn|  2925|     1298| 1122|    179|
+--------------+------+---------+-----+-------+



In [18]:
sub_events.withColumn("event_day", F.datediff(F.col("event_date"), F.lit(OBS_START))) \
    .groupBy("event_type").agg(
        F.mean("event_day").alias("avg_day"),
        F.min("event_day").alias("min_day"),
        F.max("event_day").alias("max_day"),
    ).show()

+----------+------------------+-------+-------+
|event_type|           avg_day|min_day|max_day|
+----------+------------------+-------+-------+
|    cancel| 69.50213873907383|     56|     83|
|   upgrade|41.810509554140125|      0|     83|
|     renew| 41.47015216543114|      0|     83|
| downgrade| 62.46957214988041|     42|     83|
+----------+------------------+-------+-------+



Payments EDA

In [19]:
payments.groupBy("payment_type").agg(
    F.count("payment_id").alias("count"),
    F.mean("amount_usd").alias("avg_amount"),
    F.sum("amount_usd").alias("total_amount"),
).show()

payments.groupBy("status").count().show()

# Failed/refund rate by persona
payments.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").pivot("status").agg(F.count("payment_id")).show()

+------------+-----+------------------+------------------+
|payment_type|count|        avg_amount|      total_amount|
+------------+-----+------------------+------------------+
|    day_pass|61375|3.9899999999968956|244886.24999980946|
|      in_app|82636|2.7468614163292364|226989.63999978278|
|subscription|95037|13.819035007410017|1313319.6299992257|
+------------+-----+------------------+------------------+

+--------+------+
|  status| count|
+--------+------+
|refunded|  5674|
| success|224579|
|  failed|  8795|
+--------+------+

+--------------+------+--------+-------+
|       persona|failed|refunded|success|
+--------------+------+--------+-------+
|        casual|  3023|    1774|  54744|
|      hardcore|  1394|     738|  67013|
|       regular|  2778|    1783|  89209|
|about_to_churn|  1600|    1379|  13613|
+--------------+------+--------+-------+



Features Engineering

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import datetime

spark = SparkSession.builder \
    .appName("gfn-feature-engineering") \
    .master("local[*]") \
    .getOrCreate()

users = spark.read.parquet("/home/spark/work/data/raw/users.parquet")
sessions = spark.read.parquet("/home/spark/work/data/raw/session_logs.parquet")

In [22]:
# Temporal windows
DATA_START = datetime.date(2024, 1, 1)
OBS_START = datetime.date(2024, 1, 29)   # Week 5 start (day 29)
OBS_END = datetime.date(2024, 2, 26)     # Week 9 start (exclusive)

In [24]:
obs_sessions = sessions.filter(
    (F.col("start_time").cast("date") >= F.lit(OBS_START)) &
    (F.col("start_time").cast("date") < F.lit(OBS_END))
)

obs_sessions = obs_sessions \
    .withColumn(
        "week_num",
        (F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7).cast("int") + 1
    ) \
    .withColumn(
        "duration_min",
        (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
    )

In [25]:
obs_sessions.show(5, truncate=False)

+----------+-------+-------+-------------------+--------------------------+-----------------+--------------+-------+-----------------+----------------+------------+----------------+-------------+----------------+---------+--------+------------------+
|session_id|user_id|game_id|start_time         |end_time                  |stream_resolution|avg_latency_ms|avg_fps|total_frame_drops|disconnect_count|input_lag_ms|avg_bitrate_mbps|avg_jitter_ms|packet_loss_rate|exit_type|week_num|duration_min      |
+----------+-------+-------+-------------------+--------------------------+-----------------+--------------+-------+-----------------+----------------+------------+----------------+-------------+----------------+---------+--------+------------------+
|S00526965 |U007307|G0001  |2024-01-29 00:02:00|2024-01-29 02:45:42.681432|4K               |11.8          |53.1   |7436             |0               |16.5        |50.4            |11.0         |0.0176          |normal   |1       |163.7           

In [26]:
obs_sessions.select(
    F.min("start_time").alias("earliest"),
    F.max("start_time").alias("latest"),
    F.countDistinct("user_id").alias("unique_users"),
    F.count("session_id").alias("total_sessions"),
).show(truncate=False)

+-------------------+-------------------+------------+--------------+
|earliest           |latest             |unique_users|total_sessions|
+-------------------+-------------------+------------+--------------+
|2024-01-29 00:02:00|2024-02-25 23:59:00|49869       |1290560       |
+-------------------+-------------------+------------+--------------+



In [27]:
obs_sessions.groupBy("week_num").count().orderBy("week_num").show()

+--------+------+
|week_num| count|
+--------+------+
|       1|270730|
|       2|270060|
|       3|269869|
|       4|479901|
+--------+------+



First five features

In [30]:
session_patterns = obs_sessions.groupBy("user_id", "week_num").agg(
    F.count("session_id").alias("weekly_session_count"),
    F.avg("duration_min").alias("avg_session_duration_min"),
    F.sum("duration_min").alias("total_playtime_min"),
    F.avg(
        F.when(F.hour("start_time").between(19, 23), 1).otherwise(0)
    ).alias("peak_hour_ratio"),
    F.avg(
        F.when(F.dayofweek("start_time").isin(1, 7), 1).otherwise(0)
    ).alias("weekend_ratio")
)

In [32]:
w = Window.partitionBy("user_id", "week_num").orderBy("start_time")

with_gap = obs_sessions.withColumn(
    "prev_end", F.lag("end_time").over(w)
).withColumn(
    "inter_session_gap_min",
    (F.unix_timestamp("start_time") - F.unix_timestamp("prev_end")) / 60.0
)

session_regularity = with_gap \
    .filter(F.col("inter_session_gap_min").isNotNull()) \
    .groupBy("user_id", "week_num") \
    .agg(F.stddev("inter_session_gap_min").alias("session_regularity"))

In [37]:
session_regularity.show(5, truncate=False)

+-------+--------+------------------+
|user_id|week_num|session_regularity|
+-------+--------+------------------+
|U000002|3       |NULL              |
|U000002|4       |4173.632897606291 |
|U000005|3       |1050.5348941134703|
|U000006|3       |NULL              |
|U000007|1       |NULL              |
+-------+--------+------------------+
only showing top 5 rows



In [33]:
session_features = session_patterns.join(
    session_regularity, on=["user_id", "week_num"], how="left"
).fillna(0, subset=["session_regularity"])

In [34]:
# Shape check
print(f"Rows: {session_features.count():,}")
print(f"Unique users: {session_features.select('user_id').distinct().count():,}")

# Per-persona sanity check
session_features.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").agg(
        F.avg("weekly_session_count").alias("avg_sessions"),
        F.avg("avg_session_duration_min").alias("avg_duration"),
        F.avg("session_regularity").alias("avg_regularity"),
        F.avg("peak_hour_ratio").alias("avg_peak"),
        F.avg("weekend_ratio").alias("avg_weekend"),
    ).show()

Rows: 186,486


Unique users: 49,869


+--------------+------------------+------------------+------------------+-------------------+-------------------+
|       persona|      avg_sessions|      avg_duration|    avg_regularity|           avg_peak|        avg_weekend|
+--------------+------------------+------------------+------------------+-------------------+-------------------+
|        casual| 2.301758331629524| 46.77770479067532| 655.0350766970371| 0.6915777764796353| 0.5981433238834578|
|      hardcore|15.036644624787096|145.83326571377094|  723.393299274699|0.49438740627573485|0.39869544780379973|
|       regular| 6.213659329810094| 87.60230321325885|1306.8023680403128| 0.5942082563119347| 0.5006750993466207|
|about_to_churn|  5.20176536002224| 73.24252913332028|1274.9648293613113| 0.5918450418818861| 0.5021096962340895|
+--------------+------------------+------------------+------------------+-------------------+-------------------+

